# 4.5 E) Transformaciones de Espacio

Seccion de demostraciones algebraicas (4.5.1 y 4.5.2) y comparacion de escalamientos (4.5.3) sobre la matriz imputada MICE de 4.3.

### 4.5.1 Demostracion de invarianza del centrado

Sea $X \in \mathbb{R}^{n \times d}$ con media muestral $\mu = X^T \mathbf{1}/n$ y matriz de covarianza

$$
\Sigma_X = \frac{1}{n}\sum_{i=1}^{n} (x_i - \mu)(x_i - \mu)^T
         = \frac{1}{n}X^T X - \mu\mu^T.
$$

El centrado es $Z = X - \mathbf{1}\mu^T$, donde $\mathbf{1}$ es el vector de unos. Despues de centrar la media se anula:

$$
\mu_Z = \frac{1}{n}Z^T\mathbf{1} = \frac{1}{n}\big(X^T\mathbf{1} - \mu \mathbf{1}^T\mathbf{1}\big) = \mu - \mu = 0,
$$

y cada fila cumple $z_i = x_i - \mu$. Entonces la covarianza de $Z$ es

$$
\Sigma_Z = \frac{1}{n}\sum_{i=1}^{n} (z_i - 0)(z_i - 0)^T
         = \frac{1}{n}\sum_{i=1}^{n} (x_i - \mu)(x_i - \mu)^T
         = \Sigma_X.
$$

Tomando la traza,

$$
\operatorname{tr}(\Sigma_Z) = \operatorname{tr}(\Sigma_X)
    = \frac{1}{n}\sum_{i=1}^{n}\|x_i - \mu\|^2
    = \frac{1}{n}\sum_{j=1}^{d}\sum_{i=1}^{n}(x_{ij} - \mu_j)^2,
$$

la varianza total (dispersion) del conjunto. El centrado solo traslada el origen sin modificar las desviaciones respecto de la media, por lo que la estructura de dispersion total, $\operatorname{tr}(\Sigma)$, permanece constante bajo traslacion. $\square$

In [1]:
# Verificación numérica: tr(Σ) se preserva bajo centrado
import numpy as np, pandas as pd
_df = pd.read_parquet("data/arrhythmia_imputado.parquet")
Xr = _df.drop(columns=["Class"]).to_numpy(dtype=float)
Xc = Xr - Xr.mean(axis=0)  # centrado
tr_orig = np.trace(np.cov(Xr, rowvar=False, ddof=0))
tr_cent = np.trace(np.cov(Xc, rowvar=False, ddof=0))
print(f"tr(Sigma) original:  {tr_orig:.6f}")
print(f"tr(Sigma) centrado:  {tr_cent:.6f}")
print(f"Iguales (atol=1e-10): {np.isclose(tr_orig, tr_cent, atol=1e-10)}")
del _df, Xr  # limpiar variables temporales

tr(Sigma) original:  40243.942911
tr(Sigma) centrado:  40243.942911
Iguales (atol=1e-10): True


### 4.5.2 Propagacion de covarianza $y = W^T x$

Dada la transformacion lineal $y = W^T x$ con $W$ fija y $x$ vector aleatorio de media $\mu_x$ y covarianza $\Sigma_X$, la covarianza de $y$ es

$$
\Sigma_Y = \mathbb{E}\big[(y - \mu_y)(y - \mu_y)^T\big].
$$

Por linealidad de la esperanza, la media transformada es $\mu_y = \mathbb{E}[W^T x] = W^T \mu_x$. Restando,

$$
y - \mu_y = W^T x - W^T \mu_x = W^T (x - \mu_x).
$$

Sustituyendo en la definicion y sacando $W$ (constante) fuera de la esperanza:

$$
\Sigma_Y = \mathbb{E}\Big[W^T (x - \mu_x)(x - \mu_x)^T W\Big]
         = W^T \, \mathbb{E}\big[(x - \mu_x)(x - \mu_x)^T\big]\, W
         = W^T \Sigma_X W.
$$

Esto es la regla de propagacion de covarianza: la dispersion del espacio imagen queda determinada por la matriz de covarianza del espacio original a traves del conjugado $W^T (\cdot) W$. En particular, si $W$ es ortogonal, $\operatorname{tr}(\Sigma_Y)=\operatorname{tr}(W^T\Sigma_X W)=\operatorname{tr}(\Sigma_X)$, consistente con 4.5.1. $\square$

In [2]:
# Verificación numérica: Σ_Y = W^T Σ_X W
import numpy as np, pandas as pd
_df2 = pd.read_parquet("data/arrhythmia_imputado.parquet")
_X = _df2.drop(columns=["Class"]).to_numpy(dtype=float)
_Xc = _X - _X.mean(axis=0)
rng_w = np.random.default_rng(42)
d_red = 50  # dimensión reducida
W = rng_w.standard_normal((_Xc.shape[1], d_red))
Y = _Xc @ W  # n x d_red
Sigma_X = np.cov(_Xc, rowvar=False, ddof=0)
Sigma_Y_obs = np.cov(Y, rowvar=False, ddof=0)
Sigma_Y_formula = W.T @ Sigma_X @ W
max_err = np.max(np.abs(Sigma_Y_obs - Sigma_Y_formula))
print(f"Σ_Y observada (shape {Sigma_Y_obs.shape}) vs Σ_Y = W^T Σ_X W:")
print(f"  error absoluto max: {max_err:.2e}")
print(f"  coincide (atol=1e-8): {np.allclose(Sigma_Y_obs, Sigma_Y_formula, atol=1e-8)}")
# Caso ortogonal: tr(Σ_Y) = tr(Σ_X)
d = _Xc.shape[1]
Q, _ = np.linalg.qr(rng_w.standard_normal((d, d)))
Y_ort = _Xc @ Q
tr_orig = np.trace(Sigma_X)
tr_ort = np.trace(np.cov(Y_ort, rowvar=False, ddof=0))
print(f"\nW ortogonal: tr(Σ_X)={tr_orig:.2f}, tr(Σ_Y)={tr_ort:.2f}, iguales={np.isclose(tr_orig, tr_ort)}")
del _df2, _X, _Xc  # limpiar variables temporales

Σ_Y observada (shape (50, 50)) vs Σ_Y = W^T Σ_X W:
  error absoluto max: 5.82e-11
  coincide (atol=1e-8): True

W ortogonal: tr(Σ_X)=40243.94, tr(Σ_Y)=40243.94, iguales=True


### 4.5.3 Comparacion de escalamientos: z-score vs min-max

Sobre la matriz final de trabajo se compara el numero de condicion kappa de la matriz centrada respecto de la estandarizada (z-score) y la escalada min-max. Antes se resuelve la singularidad detectada en 4.4 eliminando **una columna por direccion nula del espacio nulo (seleccion canonica, la de mayor peso en cada vector del nucleo)**. kappa se mide en la convencion unificada de la tarea: espectro de la matriz centrada (Sigma poblacional 1/n).

Ademas se cuantifica la **dimensionalidad intrinseca** (numero de autovalores de la estandarizada sobre umbral numerico y varianza explicada acumulada), cerrando la promesa de la seccion A (decision 5).


In [3]:
import numpy as np, pandas as pd

df = pd.read_parquet("data/arrhythmia_imputado.parquet")
X = df.drop(columns=["Class"]).to_numpy(dtype=float)
names = list(df.drop(columns=["Class"]).columns)
n, d = X.shape
print(f"Inicial: {n} x {d} | rango: {np.linalg.matrix_rank(X)}")

# E1) Romper las 4 direcciones nulas con seleccion CANONICA del espacio nulo (4.4.1):
# una columna por direccion, la de mayor peso en cada vector v del nucleo
Xc5 = X - X.mean(axis=0)
_, s5, Vt5 = np.linalg.svd(Xc5, full_matrices=False)
rk5 = int((s5 > np.finfo(float).eps * max(n, d) * s5[0]).sum())
N5 = Vt5[rk5:]                      # 4 vectores del nucleo
quita, restan = [], list(range(d))
for v in N5:
    j_peak = max(restan, key=lambda j: abs(v[j]))
    quita.append(names[j_peak])
    restan.remove(j_peak)
print(f"Quitadas (canonicas, una por direccion del nucleo): {quita}")

keep = [names[j] for j in restan]
Xr = X[:, restan]
print(f"Tras reducir: {Xr.shape[0]} x {Xr.shape[1]} | rango: {np.linalg.matrix_rank(Xr - Xr.mean(axis=0))}")

# Version canonicamente reducida; luego centrado / z-score / min-max
mu = Xr.mean(axis=0)
Xc = Xr - mu
Xz = (Xr - mu) / Xr.std(ddof=0, axis=0)
mn = Xr.min(axis=0); mx = Xr.max(axis=0)
range_ = mx - mn
range_[range_ == 0] = 1  # evita division por cero en columnas constantes
Xm = (Xr - mn) / range_

Inicial: 452 x 261 | rango: 257


Quitadas (canonicas, una por direccion del nucleo): ['DI_QRSA', 'V4_R_width', 'V2_n_intrinsic_deflections', 'V3_R_prime_width']
Tras reducir: 452 x 257 | rango: 257


In [4]:
# kappa (convencion unificada): espectro de la matriz CENTRADA
def kappa(M):
    Mc = M - M.mean(axis=0)
    ev = np.linalg.eigvalsh((Mc.T @ Mc) / Mc.shape[0])  # Sigma poblacional 1/n
    tol = np.finfo(float).eps * max(Mc.shape) * ev[-1]
    mask = ev > tol
    if not mask.any():
        return np.inf  # efectivamente singular
    lam_min = ev[mask][0]
    return ev[-1] / lam_min

for etiqueta, M in (("centrada", Xc), ("z-score", Xz), ("min-max", Xm)):
    kc = kappa(M)
    trd = np.trace(np.cov(M, rowvar=False, ddof=0))
    print(f"{etiqueta:8s} kappa = {kc:.3e}  (log10 = {np.log10(kc):.1f})  | tr = {trd:.2f}")
# Verificacion de la advertencia: las 5 columnas casi-lineales señaladas en la
# seccion 4.4 siguen casi determinadas por el resto de la matriz final. Se mide
# R2 de cada una contra las otras 256 (regresion lineal por minimos cuadrados).
casi = ["AVL_Q_width", "AVF_S_width", "V4_QRSTA", "V5_diphasic_P", "V5_T_ampl"]
Xcc = Xz - Xz.mean(axis=0)
print("\nR2 de las columnas casi-lineales retenidas (contra las otras 256):")
r2s = []
for c in casi:
    j = keep.index(c)
    y = Xcc[:, j]
    others = np.delete(Xcc, j, axis=1)
    coef, *_ = np.linalg.lstsq(others, y, rcond=None)
    r2 = 1 - (y - others @ coef).var() / y.var()
    r2s.append(r2)
    print(f"  {c:<18} R2 = {r2:.6f}")
print(f"R2 minimo = {min(r2s):.5f} (>= 0.99965): colinealidad fuerte retenida a proposito")


centrada kappa = 2.098e+10  (log10 = 10.3)  | tr = 40243.86
z-score  kappa = 4.526e+06  (log10 = 6.7)  | tr = 257.00
min-max  kappa = 3.027e+07  (log10 = 7.5)  | tr = 3.94

R2 de las columnas casi-lineales retenidas (contra las otras 256):
  AVL_Q_width        R2 = 0.999983


  AVF_S_width        R2 = 0.999652


  V4_QRSTA           R2 = 0.999992
  V5_diphasic_P      R2 = 0.999965


  V5_T_ampl          R2 = 0.999653
R2 minimo = 0.99965 (>= 0.99965): colinealidad fuerte retenida a proposito


**Conclusion 4.5.3.**

Reduccion previa: se removieron 4 columnas, una por direccion del nucleo de 4.4.1 (seleccion canonica por mayor peso en cada vector nulo: DI_QRSA, V4_R_width, V2_n_intrinsic_deflections y V3_R_prime_width). Es la matriz final de trabajo (452 x 257, rango pleno).

Comparacion de kappa (espectro de la matriz centrada, Sigma poblacional 1/n; convencion unificada):

| Escalamiento | kappa | log10 | tr(Sigma) |
|---|---|---|---|
| Centrada | 2.10e10 | 10.3 | 40243.86 |
| z-score | 4.53e6 | 6.7 | 257.00 |
| min-max | 3.03e7 | 7.5 | 3.94 |

- **La centrada es la peor condicionada**: las columnas quedan en unidades originales (mV, ms), con dispersiones que difieren en ordenes de magnitud.
- **z-score gana por ~3.7 ordenes** sobre la centrada: todas las columnas quedan con varianza 1 (tr(Sigma) = 257 = numero de columnas, exactamente), igualando la escala.
- **min-max queda a menos de 1 orden de z-score** pero peor: lleva cada columna a [0,1] con dispersion dependiente del rango observado; las colas extremas inflan el rango y degradan el condicionamiento (tr = 3.94, no 257).

**Advertencia honesta sobre el resultado.** kappa ~ 4.5e6 en la estandarizada sigue siendo alto: la estandarizacion iguala las varianzas marginales pero NO elimina la correlacion fuerte entre columnas (kappa de una matriz de correlacion con rho=0.99 sigue siendo ~200). Se verifica en la salida: las 5 columnas casi-lineales (AVL_Q_width, AVF_S_width, V4_QRSTA, V5_diphasic_P, V5_T_ampl) tienen R2 >= 0.99965 contra el resto de la matriz. Se conservan por fidelidad fisiologica y se deja documentado el trade-off; un paso adicional de filtro por varianza minima o blanqueamiento (PCA/whitening) reducira kappa si el problema posterior lo exige. "Mejor condicionado" aqui significa 3.7 ordenes de mejora respecto de la centrada, no un espacio isotropico.

**Dimensionalidad intrinseca:** sobre la estandarizada, los 257 autovalores estan sobre el umbral numerico y el top-20 explica solo 57.4% de la varianza (ver salida de la celda de exportacion): no procede comprimir por PCA (decision 5 de A); la varianza sigue repartida en muchas direcciones.

In [5]:
import os, json

# Matriz final (z-score) + clase; se conservan nombres de columna
X_final = pd.DataFrame(Xz, columns=keep)
out = X_final.copy()
out["Class"] = df.loc[X_final.index, "Class"].to_numpy()
out.index = df.index

os.makedirs("data", exist_ok=True)
out.to_parquet("data/arrhythmia_final.parquet", index=False)

# Dimensionalidad intrinseca de la estandarizada (cierre de la decision 5 de A)
ev_z = np.linalg.eigvalsh(np.cov(Xz, rowvar=False, ddof=0))[::-1]
tol_z = np.finfo(float).eps * max(Xz.shape) * ev_z[0]
n_sobre_tol = int((ev_z > tol_z).sum())
top20 = float(ev_z[:20].sum() / ev_z.sum())

meta = {
    "fuente": "data/arrhythmia_imputado.parquet (IterativeImputer, 4.3)",
    "dimension_final": "452 x 257 features + Class",
    "reduccion_4.4": {"quitadas": quita, "criterio": "una columna por direccion del nucleo (mayor peso SVD)", "rango_final": int(np.linalg.matrix_rank(Xr - Xr.mean(axis=0)))},
    "escalamiento_4.5": "z-score (media 0, desviacion 1 por columna)",
    "kappa_final_log10": float(np.log10(kappa(Xz))),
    "dimensionalidad_intrinseca": {"autovalores_sobre_tol": n_sobre_tol, "varianza_top20": top20},
    "casi_lineales_retenidas": ["AVL_Q_width", "AVF_S_width", "V4_QRSTA", "V5_diphasic_P", "V5_T_ampl"],
    "mu": {c: float(m) for c, m in zip(keep, mu)},
    "sigma": {c: float(s) for c, s in zip(keep, Xr.std(ddof=0, axis=0))},
    "nota": "mu/sigma se estimaron sobre el conjunto completo; F (seccion 4.6) define el protocolo sin fuga para inferencia/modelado"
}
with open("data/transformacion_metadatos.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=True, indent=2)

print(f"data/arrhythmia_final.parquet: {out.shape[0]} filas x {out.shape[1]} cols (NaN: {int(out.drop(columns=['Class']).isna().sum().sum())})")
print(f"kappa final (log10) = {meta['kappa_final_log10']:.2f}")
print(f"Dimensionalidad intrinseca: {n_sobre_tol} autovalores > tol | top-20 explica {top20:.1%}")
print(f"data/transformacion_metadatos.json: {len(meta)} claves")


data/arrhythmia_final.parquet: 452 filas x 258 cols (NaN: 0)


kappa final (log10) = 6.66
Dimensionalidad intrinseca: 257 autovalores > tol | top-20 explica 57.4%
data/transformacion_metadatos.json: 10 claves
